In [1]:
pip install mysql-connector-python SQLAlchemy pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
from sqlalchemy import create_engine

# Configuración de conexión con tu Workbench
user = "root"
password = "IRONHACK"   # ⚠️ Asegúrate de cambiar la contraseña si es diferente
host = "127.0.0.1"
port = 3306
database = "sakila"   # ⚠️ Asegúrate de haber cargado la base de datos sakila en MySQL

# Crear el motor de conexión
engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}:{port}/{database}")



In [4]:
def rentals_month(engine, month, year):
    query = f"""
        SELECT customer_id, rental_date
        FROM rental
        WHERE MONTH(rental_date) = {month}
          AND YEAR(rental_date) = {year};
    """
    df = pd.read_sql(query, engine)
    return df


In [5]:
df_may = rentals_month(engine, 5, 2005)
print(df_may.head())


   customer_id         rental_date
0          130 2005-05-24 22:53:30
1          459 2005-05-24 22:54:33
2          408 2005-05-24 23:03:39
3          333 2005-05-24 23:04:41
4          222 2005-05-24 23:05:21


In [6]:
def rental_count_month(df, month, year):
    counts = df.groupby("customer_id").size().reset_index(name=f"alquileres_{month:02d}_{year}")
    return counts


In [7]:
def compare_rentals(df1, df2):
    merged = pd.merge(df1, df2, on="customer_id", how="outer").fillna(0)
    col1 = df1.columns[1]
    col2 = df2.columns[1]
    merged["diferencia"] = merged[col2] - merged[col1]
    return merged
